# Titanic Survival Prediction — Train and Save Model

This notebook trains a Titanic classifier and saves the complete preprocessing + model pipeline as `titanic_model.pkl`.

Run all cells in Google Colab with internet access. This is a standalone example; if your instructor's notebook uses a different dataset or feature set, follow the instructor notebook for graded Task 1.


In [ ]:
%pip -q install seaborn scikit-learn pandas


In [ ]:
import pandas as pd
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## Step 1 — Load the Titanic dataset
Target `survived`: 0 = did not survive, 1 = survived.

In [ ]:
df = sns.load_dataset("titanic")
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


## Step 2 — Select features and target
We use passenger class, sex, siblings/spouses aboard, and embarkation port.

In [ ]:
features = ["pclass", "sex", "sibsp", "embarked"]
data = df[features + ["survived"]].dropna(subset=["survived"]).copy()
X = data[features]
y = data["survived"].astype(int)
print("Features:", features)
display(y.value_counts().sort_index())


## Step 3 — Split into training and test sets (80/20).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Step 4 — Preprocess and train an SVC model.

In [ ]:
numeric_features = ["pclass", "sibsp"]
categorical_features = ["sex", "embarked"]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(probability=True, random_state=42))
])
model.fit(X_train, y_train)
print("Model training completed.")


## Step 5 — Evaluate the model. Record the actual results.

In [ ]:
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


## Step 6 — Save the trained model as `titanic_model.pkl`.

In [ ]:
with open("titanic_model.pkl", "wb") as file:
    pickle.dump(model, file)
print("Saved titanic_model.pkl")


## Step 7 — Load and test the saved model.

In [ ]:
with open("titanic_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

sample = pd.DataFrame([{
    "pclass": 3, "sex": "female", "sibsp": 0, "embarked": "S"
}])
prediction = int(loaded_model.predict(sample)[0])
probability = float(loaded_model.predict_proba(sample)[0, 1])
display(sample)
print("Predicted class:", prediction, "(1 = survived, 0 = did not survive)")
print(f"Estimated survival probability: {probability:.3f}")


## Step 8 — Download the file
In Colab, open Files on the left, find `titanic_model.pkl`, click the three dots, and choose Download.